## The Problem

Every tutorial uses `large_final` (magnitude pruning). But on a pretrained ResNet-18 at 70% sparsity, `wanda` retains **5% more accuracy** than `large_final` — same sparsity, same model, just a different criterion.

fasterai provides 14 criteria. This guide shows which ones matter and when.

In [1]:
from fastai.vision.all import *
# Import criteria AFTER fastai (fastai's import * shadows fasterai's `random` criterion)
from fasterai.core.criteria import *
from fasterai.sparse.sparsifier import Sparsifier
from fasterai.sparse.sparsify_callback import SparsifyCallback
from fasterai.core.schedule import one_shot, agp

## Head-to-Head: Magnitude vs Wanda

Same pretrained ResNet-18, same Imagenette dataset, same 70% sparsity. Only the criterion changes:

In [ ]:
path = untar_data(URLs.IMAGENETTE_160)
dls = ImageDataLoaders.from_folder(path, valid='val', bs=64, item_tfms=Resize(128))

cal_data = [dls.one_batch()[0]]

# Baseline: vision_learner handles the 1000→10 class head automatically
learn_base = vision_learner(dls, resnet18, pretrained=True, metrics=accuracy)
base_acc = learn_base.validate()[1] * 100
print(f'Baseline (no pruning): {base_acc:.1f}%')

print(f'\nResNet-18 @ 70% sparsity:')
for name, crit, kw in [
    ('random',       random,       {}),
    ('large_final',  large_final,  {}),
    ('wanda',        wanda,        {'data': cal_data}),
]:
    learn = vision_learner(dls, resnet18, pretrained=True, metrics=accuracy)
    learn.model.eval()
    Sparsifier(learn.model, 'weight', 'local', crit, **kw).sparsify_model(70)
    acc = learn.validate()[1] * 100
    print(f'  {name:<15} {acc:5.1f}%  ({acc - base_acc:+5.1f})')

**Why the gap?** Magnitude pruning assumes "small weight = unimportant." But a small weight connected to a highly active channel contributes more to the output than a large weight on a dead channel. Wanda fixes this: `score = |W| × ‖X‖₂`.

## When to Use What

```
                         ┌─ Yes → wanda (best, +5% over magnitude)
Have calibration data? ─┤
                         └─ No
                              │
              ┌─ One-shot ────┤── large_final
              │               │
Pruning when? ┤               └─ During training?
              │                      │
              └─ Gradual (AGP) ──────┤── updating_movmag
                                     │
                                     └── movement (lottery ticket research)
```

## The 3 Criteria You Should Know

### `large_final` — the safe default

Keep weights with largest `|W|`. Works post-training, no data needed. Start here.

In [ ]:
# Post-training, one line:
model = resnet18(weights='DEFAULT')
Sparsifier(model, 'weight', 'local', large_final).sparsify_model(50)

# During training with SparsifyCallback:
cb = SparsifyCallback(sparsity=50, granularity='weight', context='local',
                      criteria=large_final, schedule=one_shot)

### `wanda` — the best post-training criterion

Scores `|W| × ‖X‖₂` using a few calibration batches. Consistently 3-5% better than magnitude at 50-80% sparsity.

In [ ]:
# Just need a few batches of representative data
model = resnet18(weights='DEFAULT')
cal_data = [torch.randn(8, 3, 224, 224)]
Sparsifier(model, 'weight', 'local', wanda, data=cal_data).sparsify_model(70)

### `updating_movmag` — the best during-training criterion

Scores `|W × (W − W_prev)|` — weights must be both **large** and **recently changing**. Re-evaluates importance at every pruning step, so it adapts as the network learns.

Pair with a gradual schedule (AGP or cosine):

In [ ]:
# Example: gradual pruning during training with updating_movmag
learn = vision_learner(dls, resnet18, pretrained=True, metrics=accuracy)

cb = SparsifyCallback(
    sparsity=70, granularity='weight', context='local',
    criteria=updating_movmag,  # re-evaluates at each step
    schedule=agp,              # gradual — gives movmag time to assess
)
learn.fit_one_cycle(20, 1e-3, cbs=cb)

## Full Reference

| Criterion | Score | Needs | When |
|-----------|-------|-------|------|
| `large_final` | `\|W\|` | Nothing | Post-training default |
| **`wanda`** | `\|W\| × ‖X‖₂` | Calibration data | **Best post-training** |
| **`updating_movmag`** | `\|W(W−W_{prev})\|` | Training | **Best during training** |
| `movement` | `\|W−W₀\|` | Training | Lottery ticket research |
| `magnitude_increase` | `\|W\|−\|W₀\|` | Training | Growing weights |
| `large_init_large_final` | `min(\|W\|,\|W₀\|)` | Training | Conservative |
| `squared_final` | `W²` | Nothing | ≈ large_final |
| `random` | Random | Nothing | Baseline only |
| `grad_crit` | `(W×∇W)²` | Gradients | Taylor pruning |

---

## See Also

- [Wanda Tutorial](../sparse/wanda.html) — Deep dive into activation-aware pruning
- [Schedules](../sparse/schedules.html) — AGP, cosine, and other pruning schedules
- [Criteria API](../../core/criteria.html) — Full reference for all 14 criteria